In [50]:
import pandas as pd
from get_path import get_path as gp


In [51]:
from sklearn.impute import SimpleImputer

In [52]:
from sklearn.compose import ColumnTransformer, make_column_selector

In [53]:
df = pd.read_csv(gp())

In [54]:
df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,NaN,10.38,122.80,1001.0,0.1184,0.2776,0.3001,0.14710,0.2419,0.07871,...,17.33,NaN,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,NaN,NaN,0.0869,0.07017,NaN,0.05667,...,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,NaN,0
2,19.69,21.25,130.00,1203.0,0.1096,0.1599,NaN,NaN,NaN,0.05999,...,25.53,NaN,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.1425,0.2839,0.2414,NaN,0.2597,0.09744,...,26.50,NaN,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,NaN,NaN,NaN,0.1328,0.1980,NaN,0.1809,NaN,...,16.67,152.2,1575.0,0.1374,NaN,0.4000,0.1625,0.2364,0.07678,0


In [55]:
cat_dtypes = ["object", "category", "bool"]

num_vars = df.select_dtypes(exclude=cat_dtypes).columns
cat_vars = df.select_dtypes(include=cat_dtypes).columns

/var/folders/4z/3x2hpxhx0rd0hbv2nbcjtwqw0000gn/T/ipykernel_1749/793529713.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_vars = df.select_dtypes(include=cat_dtypes).columns


In [56]:
num_vars, ("-" * 50), cat_vars

(Index(['mean radius', 'mean texture', 'mean perimeter', 'mean area',
        'mean smoothness', 'mean compactness', 'mean concavity',
        'mean concave points', 'mean symmetry', 'mean fractal dimension',
        'radius error', 'texture error', 'perimeter error', 'smoothness error',
        'compactness error', 'concavity error', 'concave points error',
        'symmetry error', 'fractal dimension error', 'worst radius',
        'worst texture', 'worst perimeter', 'worst area', 'worst smoothness',
        'worst compactness', 'worst concavity', 'worst concave points',
        'worst symmetry', 'worst fractal dimension', 'target'],
       dtype='str'),
 '--------------------------------------------------',
 Index(['area error'], dtype='str'))

# ***Using column transformer*** 
### ***Method 1:*** By listing columns to be transformed

In [57]:
cleaner = ColumnTransformer([ #argument 1 -> list of tuples, eache tupple conisting of (name of transformer, transformer object/strategy, columns to be transformed)
    ("numerical_transformer", SimpleImputer(strategy="mean"), num_vars), #numerical transformer, fill Nan values with mean of the column for each numerical column
    ("categorical_transformer", SimpleImputer(strategy="most_frequent"), cat_vars) #categorical transformer, fill Nan values with most frequent value for each categorical column
])


In [58]:
cleaner.fit_transform(df) #applies on entire dataframe

array([[14.059547717842323, 10.38, 122.8, ..., 0.1189, 0.0, 'A'],
       [20.57, 17.77, 132.9, ..., 0.08436317021276594, 0.0, 'A'],
       [19.69, 21.25, 130.0, ..., 0.08758, 0.0, 'A'],
       ...,
       [16.6, 28.08, 108.3, ..., 0.0782, 0.0, 'A'],
       [20.6, 29.33, 140.1, ..., 0.124, 0.0, 'A'],
       [7.76, 19.311829268292684, 47.92, ..., 0.07039, 1.0, 'A']],
      shape=(569, 31), dtype=object)

### ***Method 2:*** By giving indices of columns to be transformed

In [59]:
numerical_indices = df.columns.get_indexer(num_vars)
categorical_indices = df.columns.get_indexer(cat_vars)

In [60]:
numerical_indices, categorical_indices

(array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]),
 array([13]))

In [61]:
cleaner = ColumnTransformer([
    ("numerical_transformer", SimpleImputer(strategy="mean"), numerical_indices),
    ("categorical_transformer", SimpleImputer(strategy="most_frequent"), categorical_indices)
])

In [62]:
cleaner.fit_transform(df)

array([[14.059547717842323, 10.38, 122.8, ..., 0.1189, 0.0, 'A'],
       [20.57, 17.77, 132.9, ..., 0.08436317021276594, 0.0, 'A'],
       [19.69, 21.25, 130.0, ..., 0.08758, 0.0, 'A'],
       ...,
       [16.6, 28.08, 108.3, ..., 0.0782, 0.0, 'A'],
       [20.6, 29.33, 140.1, ..., 0.124, 0.0, 'A'],
       [7.76, 19.311829268292684, 47.92, ..., 0.07039, 1.0, 'A']],
      shape=(569, 31), dtype=object)

### ***Method 3:*** By passing indices(or names, same syntax) of ONLY the columns we want to edit

In [63]:
wanted_num_transformed = numerical_indices[0:3]
wanted_num_transformed

array([0, 1, 2])

In [64]:
cleaner = ColumnTransformer([
    ("numerical_transformer", SimpleImputer(strategy="mean"), wanted_num_transformed),
    ("categorical_transformer", SimpleImputer(strategy="most_frequent"), categorical_indices)
])

In [65]:
cleaned = cleaner.fit_transform(df)
cleaned

array([[14.059547717842323, 10.38, 122.8, 'A'],
       [20.57, 17.77, 132.9, 'A'],
       [19.69, 21.25, 130.0, 'A'],
       ...,
       [16.6, 28.08, 108.3, 'A'],
       [20.6, 29.33, 140.1, 'A'],
       [7.76, 19.311829268292684, 47.92, 'A']],
      shape=(569, 4), dtype=object)

### The shape indicates that only 4 features are there. Thus, here, the columns to which a transformer has not been specified, have been dropped.

### ***Ensuring that even untransformed columns remain***

In [66]:
cleaner = ColumnTransformer([
    ("numerical_transformer", SimpleImputer(strategy="mean"), wanted_num_transformed),
    ("categorical_transformer", SimpleImputer(strategy="most_frequent"), categorical_indices)
], remainder="passthrough")

In [67]:
cleaner.fit_transform(df)

array([[14.059547717842323, 10.38, 122.8, ..., 0.4601, 0.1189, 0.0],
       [20.57, 17.77, 132.9, ..., 0.275, nan, 0.0],
       [19.69, 21.25, 130.0, ..., 0.3613, 0.08758, 0.0],
       ...,
       [16.6, 28.08, 108.3, ..., 0.2218, 0.0782, 0.0],
       [20.6, 29.33, 140.1, ..., 0.4087, 0.124, 0.0],
       [7.76, 19.311829268292684, 47.92, ..., 0.2871, 0.07039, 1.0]],
      shape=(569, 31), dtype=object)

### Now, shape indicates that original untransformed features have not been dropped

# ***Autonomous Column Selection:*** **`make_column_selector()`**

In [68]:
cleaner = ColumnTransformer([
    # First transformer
    ("numerical_transformer",
     SimpleImputer(strategy="mean"), 
     #instead of using variable, use column selector
     make_column_selector(dtype_exclude=cat_dtypes)), #implies that it removes all given data types from input dataframe

    ("categorical_transformer", 
     SimpleImputer(strategy="most_frequent"),
     #same thing, except we include those dtypes
     make_column_selector(dtype_include=cat_dtypes))
], remainder="passthrough")

cleaner.fit_transform(df)

array([[14.059547717842323, 10.38, 122.8, ..., 0.1189, 0.0, 'A'],
       [20.57, 17.77, 132.9, ..., 0.08436317021276594, 0.0, 'A'],
       [19.69, 21.25, 130.0, ..., 0.08758, 0.0, 'A'],
       ...,
       [16.6, 28.08, 108.3, ..., 0.0782, 0.0, 'A'],
       [20.6, 29.33, 140.1, ..., 0.124, 0.0, 'A'],
       [7.76, 19.311829268292684, 47.92, ..., 0.07039, 1.0, 'A']],
      shape=(569, 31), dtype=object)